# Data Loading

Verify the two original UCI source files, inspect the encounter schema, and summarize patient coverage and original target labels. The mapping file is a reference for interpreting source codes.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

working_directory = Path.cwd().resolve()
project_root = next(
    directory
    for directory in (working_directory, *working_directory.parents)
    if (directory / "src" / "paths.py").is_file() and (directory / "requirements.txt").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.preparation.data_loading import (
    load_ids_mapping,
    load_raw_encounters,
    profile_raw_columns,
    summarize_raw_encounters,
)

## 1. Source Integrity

The loaders verify the exact SHA-256 digests, column order, and row counts. Both CSV files are read as text so that identifiers, empty fields, and literal markers retain their original representation.

In [ ]:
encounters = load_raw_encounters()
mapping = load_ids_mapping()

files = pd.DataFrame([
    {
        "Source file": table.path.name,
        "Data rows": len(table.frame),
        "Columns": len(table.frame.columns),
        "Bytes": table.size_bytes,
        "SHA-256": table.sha256,
    }
    for table in (encounters, mapping)
])
with pd.option_context("display.max_colwidth", None):
    display(files)

## 2. Encounter Coverage and Original Labels

Each row has a unique encounter ID. Patient IDs can repeat across encounters. The source labels retain their three original categories.

In [ ]:
summary = summarize_raw_encounters(encounters)
print(f"Encounters: {summary['encounters']:,}")
print(f"Unique patients: {summary['patients']:,}")
print(f"Patients with multiple encounters: {summary['patients_with_multiple_encounters']:,}")

labels = pd.Series(summary["target_counts"], name="Encounters")
label_table = labels.rename_axis("Original label").to_frame()
label_table["Share"] = labels.div(summary["encounters"]).map("{:.2%}".format)
display(label_table)

## 3. Encounter Schema and Literal Markers

All loaded values are strings. Logical types describe the source fields, including integer counts and categorical codes. The marker columns count distinct source representations. The string `None` in the laboratory fields is preserved as a recorded category.

In [ ]:
profile = profile_raw_columns(encounters)
display(profile.rename(columns={"column": "Column", "logical_type": "Logical type", "": "Empty"}))